**☀️ Welcome to ParkerNet Version 1.1 Notebook 1 of 3: Training Pipeline**

This notebook reproduces the training steps used to train the base model **ParkerNet Version 1.0**, which classifies solar wind switchbacks using a CNN + BiLSTM architecture.

### What This Notebook Does

With this notebook, you will be able to:

1. **Train ParkerNet** using three defined splits: **Split M**, **Split N**, and **Split P**, each with configurable random seeds  
2. **Save model weights** after training  
3. **Evaluate ParkerNet** on the validation and/or test set and generate performance metrics (e.g., **precision-recall curves**)  
4. **Run inference** on a predefined test set

---


**Data Sources:**  
ParkerNet v1.0 is trained using data from:
- **FIELDS (MAG)**: Level 2 data at 4s cadence, downsampled to 0.8738s (to match the lowest resolution in SPC data)
- **SWEAP-SPC**: Level 3 data (RTN coordinates with global quality flag)

Both datasets are available from [CDAWeb](https://cdaweb.gsfc.nasa.gov/), and were time-synchronized to create the input dataset for the network.

**SPC Data Notes:**  
SPC data contains known issues with **data gaps** related to **flow angle contamination**, and **Various types of noise**. The flow angle measurement impairment is due to a known contamination issue in Vt.  We recommend referring to the [official SWEAP SPC data release notes](http://sweap.cfa.harvard.edu/spc_data_release_notes.pdf) for encounter-specific details. Additionally, within the data release notes, refer to the reduced data quality table.

---

###  Preprocessing and Imputation Strategy:

1. **Large Gaps Filtered Out**  
   Hour-long windows with more than **25 minutes** of SPC data missing were **excluded**. Data gaps/missing data appear as a fill value in SPC data, and are accompanied by the global quality flag. If all plasma variables are missing, the flag is set to 1. Good data is accompanied by a flag of 0.

2. **Small Gaps Interpolated**  
   Gaps shorter than 25 minutes were interpolated using the **nearest-neighbor method**, which approximates each missing value by the closest known data point.


3. **Partial Component Gaps (Common in E6 onwards)**  
   Starting from Encounter 6, instances were found where the global data quality flag indicates a "good" measurement (`global quality flag = 0`), but **V<sub>t</sub> and V<sub>n</sub> are missing**, while **V<sub>r</sub> remains valid**.  
   - In such cases, **V<sub>t</sub> and V<sub>n</sub> become forward-filled**
   - These gaps occur most often in **V<sub>t</sub>** during **Encounter 6** in the dates chosen for our dataset.

4. **Contamination-Aware Training Strategy**  
   Because ParkerNet is designed to be robust to known SPC velocity artifacts:
   - **Training data** includes a small fraction (~4%) where all plasma variables are forward-filled
   - **Validation data** includes ~14% of V<sub>t</sub> forward-filled
   - **Test data (E7)** includes stretches where **all plasma variables are forward-filled**

   This controlled exposure helps the model **learn to downweight unreliable features** (especially V<sub>t</sub>), and instead rely more on magnetic field features (e.g., B<sub>r</sub>, B<sub>t</sub>, B<sub>n</sub>) or remaining plasma channels.

---

**Data File Provided:**  
The dataset `PSP_E1toE7_july23_nonoise.txt` has already undergone all preprocessing steps described above.


### ⚠️ Important Requirement

**All datasets must match the exact time resolution and input variable structure used by ParkerNet**.  
If you plan to test ParkerNet on new data (e.g., different cadence or input sources), ensure proper formatting and synchronization — otherwise, the model will not run correctly. Make sure to match the order in which variables are fed into the network.

---

**Human-in-the-Loop (HITL) Training Strategy:**  
ParkerNet has been trained with an iterative human-in-the-loop (HITL) training strategy. Starting with a small number of visually identified label time steps from Encounter 1, the model was trained, then used to predict on the next day. Obvious false positives and false negatives were manually corrected, and the newly labeled data was added to the training set for retraining. This process was repeated across subsequent time periods and encounters, gradually expanding the labeled fraction to ~12%. After this iterative labeling phase, the full labeled dataset was frozen, and the final ParkerNet model was selected and tuned. Positive class weighting in the loss function was updated at each iteration to reflect the current class balance. This approach enabled efficient model refinement without requiring full manual annotation of the dataset.


The training dataset provided (`PSP_E1toE7_July23_nonoise.txt`) has already undergone the **iterative human-in-the-loop (HITL) labeling process** and represents the final frozen dataset.  
This notebook **only recreates the final supervised training step**, not the full HITL labeling loop.

---

### Environment & Packages

This notebook was developed and tested on a **Google Colab Pro instance with an A100 GPU**.  
A `requirements.txt` is **not provided**, due to compatibility issues with high-compute environments.

However, **exact package versions** are printed at the top of this notebook to help ensure reproducibility.

---

### Required Files

Please ensure the following file is present in your working directory:

- `PSP_E1toE7_July23_nonoise.txt`
This file can be accessed at: [![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.14902750.svg)](https://doi.org/10.5281/zenodo.14902750)



**🛠️NOTE TO USER**:We adjusted the sequence creating function code slightly here, in the current sequence creation function the last window is missing. To fix this, simple do len(dataset) - time_steps + 1

This is a very minor adjustment, and does not change our conclusions.






# ====== Load Packages ======


________________________________________________________________________________________________________________________________________________________________

We will start by loading all the packages we will need for this project.

In [1]:
import pandas as pd
import os
import numpy as np
import keras
from keras import backend as K
from keras.models import Sequential
from keras.layers import Conv1D, MaxPooling1D,Activation, Dense, Dropout, Flatten, TimeDistributed, Bidirectional, LSTM, GlobalMaxPool1D, GlobalAveragePooling1D
from tensorflow.keras.layers import  Input
from keras.optimizers import Adam
import matplotlib.pyplot as plt
from time import time
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_curve
from sklearn.metrics import roc_auc_score, roc_curve, f1_score, precision_score, recall_score, accuracy_score, auc
from sklearn.metrics import precision_recall_curve, auc
from keras import initializers
import tensorflow as tf
import random

In [ ]:
# Print versions
print("Versions of installed libraries:")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"keras: {keras.__version__}")
print(f"tensorflow: {tf.__version__}")
print(f"matplotlib: {plt.matplotlib.__version__}")
print(f"seaborn: {sns.__version__}")
print(f"scikit-learn: {sns.__version__}")

Versions of installed libraries:
pandas: 2.2.2
numpy: 1.26.4
keras: 3.8.0
tensorflow: 2.18.0
matplotlib: 3.10.0
seaborn: 0.13.2
scikit-learn: 0.13.2


When running this notebook locally, ignore the google drive specific commands and simply read in the provided .txt files needed from your local folder.

In [2]:
from google.colab import drive
drive.mount('/content/MyDrive')

Mounted at /content/MyDrive


In [3]:
cd MyDrive/MyDrive

/content/MyDrive/MyDrive


# ====== Load data ======

Loading in the file that will be used for training and validation.

In [5]:
df = pd.read_csv('PSP_E1toE7_July23_nonoise.txt', sep = "\t", parse_dates=['Datetime'],
                 infer_datetime_format=True,
                 index_col='Datetime',)

Class= df['Class'].astype('int')
df.Class = df.Class.replace({True: 1, False: 0})
df.pop("indices")
df.pop("Encounter")
X_HCI_train = df.pop("X_HCI")
Y_HCI_train = df.pop("Y_HCI")
Z_HCI_train = df.pop("Z_HCI")
#ProtonDensity_train = df.pop("ProtonDensity")
df.pop("Dist")
#df.pop("Vmag")
#df.pop("Bmag")
#df.pop("V_nr")
#df.pop("B_t")
#df.pop("B_n")
#df.pop("V_n")
#df.pop("V_t")
df.head()

<ipython-input-5-1617452364>:1: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df = pd.read_csv('PSP_E1toE7_July23_nonoise.txt', sep = "\t", parse_dates=['Datetime'],
<ipython-input-5-1617452364>:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.Class = df.Class.replace({True: 1, False: 0})


,B_r,Bmag,B_t,B_n,V_r,V_t,V_n,Vmag,V_nr,ProtonDensity,Class
Datetime,,,,,,,,,,,
2018-11-04 00:00:00.000,-66.058226,70.255204,-4.140612,-23.557585,267.886,-28.3115,-39.9285,272.321015,48.947177,391.665,0
2018-11-04 00:00:00.873,-65.699724,70.171543,-3.381591,-24.416317,263.905,-69.5283,-43.8579,276.411919,82.205230,536.104,0
2018-11-04 00:00:01.747,-66.442084,70.452417,-8.469396,-21.846322,266.871,-41.5544,-24.7354,271.217143,48.359158,367.138,0
2018-11-04 00:00:02.621,-65.771892,70.127234,-15.798794,-18.500954,260.689,-57.2835,-41.0414,270.045460,70.468403,504.809,0
2018-11-04 00:00:03.495,-64.232595,70.390542,-18.842718,-21.770487,267.265,-26.0570,-45.9163,272.429540,52.794639,394.882,0


**IMPORTANT: All dataframes MUST have the exact number of columns and variables in the same order.**

# ====== Create Data Splits : Train -Val - Test ======

Splitting the dataset into various train-validation-test splits.

# **Split M**



| **Split M has data from the following dates** |
|---------------------------|
| 2018-11-04 |
| 2018-11-05 |
| 2018-11-06 |
| 2018-11-07 |
| 2018-11-08 |
| 2019-04-02 |
| 2019-04-03 |
| 2019-04-04 |
| 2019-04-05 |
| 2019-04-06 |
| 2019-04-07 |
| 2019-08-26 |
| 2019-08-27 |
| 2019-08-28 |
| 2019-08-29 |
| 2019-08-30 |
| 2019-08-31 |
| 2020-01-27 |
| 2020-01-28 |
| 2020-01-29 |
| 2020-01-31 |
| 2020-02-01 |


In [6]:
#test M`, train: E1 to E4, val:E5-E6 EOD Sept 23, test 1:E6 Sept 24 + E7
Xtrain = df.iloc[0:1615023,:-1] #everything but the last column (split1)
ytrain = df.iloc[0:1615023,-1]#only pick the last column
Xval = df.iloc[1615023:1788065,:-1] #everything but the last column (split 1)
yval = df.iloc[1615023:1788065,-1]#only pick the last column
Xtest1 = df.iloc[1788065:,:-1] #everything but the last column
ytest1 = df.iloc[1788065:,-1]#only pick the last column

# **Split N**

| **Split N has data from the following dates** |
|---------------------------|
| 2018-11-04 |
| 2018-11-05 |
| 2018-11-06 |
| 2018-11-07 |
| 2018-11-08 |
| 2019-04-02 |
| 2019-04-03 |
| 2019-04-04 |
| 2019-04-05 |
| 2019-04-06 |
| 2019-04-07 |
| 2019-08-26 |
| 2019-08-27 |
| 2019-08-28 |
| 2019-08-29 |
| 2019-08-30 |
| 2019-08-31 |


In [ ]:
#test N`, train: E1 to E3, val:E4-E6 EOD Sept 23, test 1:E6 Sept 24 + E7
Xtrain = df.iloc[0:1338985,:-1] #everything but the last column (split1)
ytrain = df.iloc[0:1338985,-1]#only pick the last column
Xval = df.iloc[1338985:1788065,:-1] #everything but the last column (split 1)
yval = df.iloc[1338985:1788065,-1]#only pick the last column
Xtest1 = df.iloc[1788065:,:-1] #everything but the last column
ytest1 = df.iloc[1788065:,-1]#only pick the last column

# **Split P**

| **Split N has data from the following dates** |
|---------------------------|
| 2019-08-26 |
| 2019-08-27 |
| 2019-08-28 |
| 2019-08-29 |
| 2019-08-30 |
| 2019-08-31 |
| 2020-01-27 |
| 2020-01-28 |
| 2020-01-29 |
| 2020-01-31 |
| 2020-02-01 |

   

In [ ]:
#test P`, train: E3 to E4, val:E5-E6 EOD SEPT 23, test 1:E6 Sept 24 + E7
Xtrain = df.iloc[786732:1615023,:-1] #everything but the last column (split1)
ytrain = df.iloc[786732:1615023,-1]#only pick the last column
Xval = df.iloc[1615023:1788065,:-1] #everything but the last column (split 1)
yval = df.iloc[1615023:1788065,-1]#only pick the last column
Xtest1 = df.iloc[1788065:,:-1] #everything but the last column
ytest1 = df.iloc[1788065:,-1]#only pick the last column

Quick check to see the amount of positive (switchback) cases in each split

In [ ]:
neg, pos = np.bincount(ytrain)
total = neg + pos
print('Examples in Train Set:\n    Total: {}\n    Positive: {} ({:.2f}% of total)\n'.format(
    total, pos, 100 * pos / total))

neg, pos = np.bincount(yval)
total = neg + pos
print('Examples in Val Set:\n    Total: {}\n    Positive: {} ({:.2f}% of total)\n'.format(
    total, pos, 100 * pos / total))

neg, pos = np.bincount(ytest1)
total = neg + pos
print('Examples in Test set 1 Set:\n    Total: {}\n    Positive: {} ({:.2f}% of total)\n'.format(
    total, pos, 100 * pos / total))

neg, pos = np.bincount(ytest2)
total = neg + pos
print('Examples in Test set 2 Set:\n    Total: {}\n    Positive: {} ({:.2f}% of total)\n'.format(
    total, pos, 100 * pos / total))

Examples in Train Set:
    Total: 1615023
    Positive: 195085 (12.08% of total)

Examples in Val Set:
    Total: 173042
    Positive: 45574 (26.34% of total)

Examples in Test set 1 Set:
    Total: 230718
    Positive: 56357 (24.43% of total)

Examples in Test set 2 Set:
    Total: 197758
    Positive: 30622 (15.48% of total)



# ====== Data Normalization ======

Z-score normalization. Note that the mean and std of the TRAIN SET is used for all sets. This is to avoid data leakage; the network must not know anything about future information.

In [9]:
train_mean = Xtrain.mean()
train_std = Xtrain.std()

train_df = (Xtrain - train_mean) / train_std
val_df = (Xval - train_mean) / train_std
test_df = (Xtest1 - train_mean) / train_std



# ====== Create Data Sequences ======


### Splitting Data into Sequences for LSTM Training

To train ParkerNet's LSTM-based model, we need to convert our time series data into overlapping sequences. Each sequence represents a fixed-length window of consecutive timesteps, allowing the model to learn temporal patterns from the input features.  After hyperparameter tuning, the best sequence length was found to be 50.

The function `split_sequences_nolags` performs this transformation:

#### Function Purpose

- Convert a continuous dataset into many overlapping sequences.
- Each sequence has a fixed length (`time_steps`).
- Used for **per-timestep sequence labeling** — for example, identifying whether each time step in the sequence is part of a switchback.

####  Inputs

- `dataset`: A pandas DataFrame of input features, shape `(total_timesteps, num_features)`
- `labels`: A pandas Series or DataFrame of binary ground truth labels aligned with the dataset
- `time_steps`: Integer specifying the desired sequence length (e.g., 50)

####  Outputs

- `data_X`: NumPy array of input sequences, shape `(num_sequences, time_steps, num_features)`
- `data_Y`: NumPy array of label sequences, shape `(num_sequences, time_steps)`

#### How It Works

- The function uses a **sliding window** of size `time_steps` to extract overlapping sequences from the dataset.
- For each step, it extracts:
  - A slice of `dataset[i : i + time_steps]` → input features
  - A slice of `labels[i : i + time_steps]` → labels for each time step
- The window advances by 1 each time.

### Why We Use Overlapping Sequences

With the split_sequencr_nolags function, we create overlapping sequences of 50 time steps for training and prediction. This means each time step appears in multiple windows, allowing the model to learn from different temporal contexts.

**Advantages of overlapping sequences:**
- Each time step is seen in multiple contexts, improving model robustness.
- Enables smoother, more stable predictions by averaging across overlaps.
- Improves sensitivity to short or subtle structures in the time series.

*Note that the overlap window can move forward by 5 or 10 time steps to improve speed if needed (adjust the code).

In contrast, non-overlapping sequences are faster and less redundant, but risk missing fine-scale transitions and may produce coarser, noisier outputs.

for our goal of fine-resolution sequence labeling, overlapping sequences are the preferred choice here. Note that it does not mean non-overlapping sequences can't be used, simply edit the function below to have no overlap. Then during prediction since each time step appears in only one sequence, you do **not** need to average across time steps. Instead, you can remove the single output dimension using `np.squeeze(..., axis=2)` to convert the model output from shape `(num_sequences, 50, 1)` to `(num_sequences, 50)`. You can then flatten the predictions using `.flatten()` to recover a 1D array representing the full length of the test or prediction set. Each of these predictions now corresponds directly to a unique, non-overlapping segment of the time series.  



In [7]:
def split_sequences_nolags(dataset,labels, time_steps):
    data_X, data_Y = [], []
    for i in range(len(dataset)-time_steps):
        a = dataset.iloc[i:(i+time_steps)]
        data_X.append(a)
        data_Y.append(labels.iloc[i:i + time_steps])
    return np.array(data_X), np.array(data_Y)

**⚠️NOTE TO USER**:We adjusted the code slightly here, in the current sequence creation function the last window is missing. To fix this, simple do len(dataset) - time_steps + 1

This is a very minor adjustment, and does not change our conclusions.

In [60]:
#Here is what you would use for no overlaps.
#def split_sequences_no_overlap(dataset, labels, time_steps):
    #data_X, data_Y = [], []
    #for i in range(0, len(dataset) - time_steps + 1, time_steps):  # Step = time_steps
        #a = dataset.iloc[i:(i + time_steps)]
        #b = labels.iloc[i:(i + time_steps)]
        #data_X.append(a)
        #data_Y.append(b)
    #return np.array(data_X), np.array(data_Y)

In [10]:
train_features_new, train_labels_new = split_sequences_nolags(train_df,ytrain,50) #originally 343, previously:90, tried: 72, 25, current: 50
val_features_new, val_labels_new = split_sequences_nolags(val_df,yval,50)
test_features_new, test_labels_new = split_sequences_nolags(test_df,ytest1,50)


n_timesteps, n_features, n_outputs = train_features_new.shape[1],train_features_new.shape[2],train_labels_new.shape[1]
train_labels_new= train_labels_new.astype('float64')
val_labels_new = val_labels_new.astype('float64')
test_labels_new = test_labels_new.astype('float64')

# ====== Setting up ParkerNet ======

Custom Binary Cross Entropy Loss Function. This modifies the regular binary cross entropy function to weigh the positive class more.

In [11]:
POS_WEIGHT = 200
POS_WEIGHT = POS_WEIGHT

def weighted_binary_crossentropy(target, output):
    """
    Weighted binary crossentropy between an output tensor
    and a target tensor. POS_WEIGHT is used as a multiplier
    for the positive targets.

    Combination of the following functions:
    * keras.losses.binary_crossentropy
    * keras.backend.tensorflow_backend.binary_crossentropy
    * tf.nn.weighted_cross_entropy_with_logits
    """
    # transform back to logits
    _epsilon = tf.convert_to_tensor(tf.keras.backend.epsilon(), output.dtype.base_dtype)
    output = tf.clip_by_value(output, _epsilon, 1 - _epsilon)
    output = tf.math.log(output / (1 - output))
    loss = tf.nn.weighted_cross_entropy_with_logits(labels=target, logits=output, pos_weight=POS_WEIGHT)

    return tf.reduce_mean(loss, axis=-1)

Metrics for evaluation when training

In [12]:
METRICS = [
    keras.metrics.Precision(name = 'precision'),
    keras.metrics.Recall(name = 'recall')
]

METRICS1 = [
      keras.metrics.TruePositives(name='tp'),
      keras.metrics.FalsePositives(name='fp'),
      keras.metrics.TrueNegatives(name='tn'),
      keras.metrics.FalseNegatives(name='fn'),
      keras.metrics.BinaryAccuracy(name='accuracy'),
      keras.metrics.Precision(name='precision'),
      keras.metrics.Recall(name='recall'),
      keras.metrics.AUC(name='auc'),
      keras.metrics.AUC(name='prc', curve='PR'), # precision-recall curve
]

# ====== Training Section ======


This section trains the final models for the various splits. The best POS weight, learning rate, layer numbers, CNN kernel size, LSTM neuron numbers etc were determined via hyperparameter tuning using Optuna.
Note to user: The MaxPooling layer is a placeholder, see code comment below.

TRAINING SPLIT M

In [ ]:
#SPLIT M

# Set seed for reproducibility, remember to set seed IN THE CELL YOU ARE RUNNING THE TRAINING.
my_seed = 3364
np.random.seed(my_seed)
random.seed(my_seed)
tf.random.set_seed(my_seed)
keras.utils.set_random_seed(my_seed)

initializer_He = keras.initializers.HeNormal(seed=my_seed)
initializer_Glorot = keras.initializers.GlorotUniform(seed=my_seed)
POS_WEIGHT = 8

# Define the model
model4 = Sequential([
    Input(shape=(n_timesteps, n_features)),  # Explicit input layer

    Conv1D(filters=32, kernel_size=5, padding='same', activation='relu', kernel_initializer=initializer_He),
    MaxPooling1D(1), #ℹ️ Placeholder: this layer currently has no effect due to pool size of 1. MaxPooling is under investigation to assess whether it excessively smooths data.This layer can be updated or removed as experiments evolve.
    Dropout(0.2),

    Conv1D(filters=64, kernel_size=3, padding='same', activation='relu', kernel_initializer=initializer_He),
    MaxPooling1D(1),
    Dropout(0.5),

    Conv1D(filters=32, kernel_size=5, padding='same', activation='relu', kernel_initializer=initializer_He),
    MaxPooling1D(1),

    Bidirectional(LSTM(24, return_sequences=True, kernel_initializer=initializer_Glorot)),
    Dropout(0.3),

    Bidirectional(LSTM(25, return_sequences=True, kernel_initializer=initializer_Glorot)),
    Dropout(0.4),

    Bidirectional(LSTM(45, return_sequences=True, kernel_initializer=initializer_Glorot)),

    Dense(1, activation='sigmoid')
])

# Compile the model
model4.compile(
    loss=weighted_binary_crossentropy,
    optimizer=Adam(learning_rate=4e-07),
    metrics=METRICS1
)

# Define callbacks
cb1_model4 = keras.callbacks.ModelCheckpoint(
    "ParkerNet_ApJS_trialtest.keras", save_best_only=True, monitor="val_loss", mode="min"
)
cb2_model4 = keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)

# UNCOMMENT TO Train the model
history_model4 = model4.fit(
    train_features_new, train_labels_new,
    epochs=60,
    batch_size=1024,
    validation_data=(val_features_new, val_labels_new),
    callbacks=[cb1_model4, cb2_model4],
    shuffle=False,
    verbose=1
)

In [14]:
model4.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 50, 32)         │         1,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 50, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 50, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 50, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 50, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 50, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 50, 32)         │        10,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 50, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 50, 48)         │        10,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 50, 48)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 50, 50)         │        14,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 50, 50)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 50, 90)         │        34,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 50, 1)          │            91 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 78,507 (306.67 KB)

 Trainable params: 78,507 (306.67 KB)

 Non-trainable params: 0 (0.00 B)

In [16]:
print(model4.input_shape)
print(model4.output_shape)

(None, 50, 10)
(None, 50, 1)


In [ ]:
#model4.save('ParkerNet_date_splitM_seed3364.keras') #uncomment for saving model and add the split and date for the date which you ran the model

In [ ]:
#use this if you want to save training history into a .csv file
hist_df = pd.DataFrame(history_model4.history)
hist_csv_file = 'history_date_splitM_seed3364.csv'
with open(hist_csv_file, mode='w') as f:
    hist_df.to_csv(f)

TRAINING SPLIT N

In [ ]:
#splitN
# Set seed for reproducibility
my_seed = 400
np.random.seed(my_seed)
random.seed(my_seed)
tf.random.set_seed(my_seed)
keras.utils.set_random_seed(my_seed)

initializer_He = keras.initializers.HeNormal(seed=my_seed)
initializer_Glorot = keras.initializers.GlorotUniform(seed=my_seed)
POS_WEIGHT = 8

# Define the model
model4 = Sequential([
    Input(shape=(n_timesteps, n_features)),  # Explicit input layer

    #Conv1D(filters=32, kernel_size=5, padding='same', activation='relu'),
    Conv1D(filters=32, kernel_size=5, padding='same', activation='relu', kernel_initializer=initializer_He),
    MaxPooling1D(1),
    Dropout(0.2),

    Conv1D(filters=64, kernel_size=3, padding='same', activation='relu', kernel_initializer=initializer_He),
    MaxPooling1D(1),
    Dropout(0.5),

    Conv1D(filters=32, kernel_size=5, padding='same', activation='relu', kernel_initializer=initializer_He),
    MaxPooling1D(1),

    Bidirectional(LSTM(24, return_sequences=True, kernel_initializer=initializer_Glorot)),
    Dropout(0.3),

    Bidirectional(LSTM(25, return_sequences=True, kernel_initializer=initializer_Glorot)),
    Dropout(0.4),

    Bidirectional(LSTM(45, return_sequences=True, kernel_initializer=initializer_Glorot)),

    Dense(1, activation='sigmoid')
])

# Compile the model
model4.compile(
    loss=weighted_binary_crossentropy,
    optimizer=Adam(learning_rate=4e-07),
    metrics=METRICS1
)

# Define callbacks
cb1_model4 = keras.callbacks.ModelCheckpoint(
    "ParkerNet_splitN_test.keras", save_best_only=True, monitor="val_loss", mode="min"
)
cb2_model4 = keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)

# Train the model
history_model4 = model4.fit(
    train_features_new, train_labels_new,
    epochs=5,
    batch_size=1024,
    validation_data=(val_features_new, val_labels_new),
    callbacks=[cb1_model4, cb2_model4],
    shuffle=False,
    verbose=1
)


TRAINING SPLIT P

In [ ]:
#Split P
# Set seed for reproducibility
my_seed = 1953
np.random.seed(my_seed)
random.seed(my_seed)
tf.random.set_seed(my_seed)
keras.utils.set_random_seed(my_seed)

initializer_He = keras.initializers.HeNormal(seed=my_seed)
initializer_Glorot = keras.initializers.GlorotUniform(seed=my_seed)
POS_WEIGHT = 8

# Define the model
model4 = Sequential([
    Input(shape=(n_timesteps, n_features)),  # Explicit input layer

    #Conv1D(filters=32, kernel_size=5, padding='same', activation='relu'),
    Conv1D(filters=32, kernel_size=5, padding='same', activation='relu', kernel_initializer=initializer_He),
    MaxPooling1D(1),
    Dropout(0.2),

    Conv1D(filters=64, kernel_size=3, padding='same', activation='relu', kernel_initializer=initializer_He),
    MaxPooling1D(1),
    Dropout(0.5),

    Conv1D(filters=32, kernel_size=5, padding='same', activation='relu', kernel_initializer=initializer_He),
    MaxPooling1D(1),

    Bidirectional(LSTM(24, return_sequences=True, kernel_initializer=initializer_Glorot)),
    Dropout(0.3),

    Bidirectional(LSTM(25, return_sequences=True, kernel_initializer=initializer_Glorot)),
    Dropout(0.4),

    Bidirectional(LSTM(45, return_sequences=True, kernel_initializer=initializer_Glorot)),

    Dense(1, activation='sigmoid')
])

# Compile the model
model4.compile(
    loss=weighted_binary_crossentropy,
    optimizer=Adam(learning_rate=1e-06),
    metrics=METRICS1
)

# Define callbacks
cb1_model4 = keras.callbacks.ModelCheckpoint(
    "ParkerNet_splitP_test.keras", save_best_only=True, monitor="val_loss", mode="min"
)
cb2_model4 = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)

# Train the model
history_model4 = model4.fit(
    train_features_new, train_labels_new,
    epochs=40,
    batch_size=1024,
    validation_data=(val_features_new, val_labels_new),
    callbacks=[cb1_model4, cb2_model4],
    shuffle=False,
    verbose=1
)


# ====== Evaluation Section ======

Predict on the test set

###  Prediction Strategy and Output Interpretation

ParkerNet is a **sequence labeling model**, meaning it outputs one probability per time step for each input sequence. However, because the input is made up of overlapping windows, we compute a **single prediction score per sequence window** to reduce redundancy and simplify evaluation.

#### 1. Overlapping Input Windows
The input data is divided into overlapping sequences of **50 time steps**, each with **10 input features**. The shape of each input sample is: (None,50, 10)

#### 2. Per-Time-Step Model Output
The model outputs a sigmoid probability at each time step, giving an output shape: (num_sequences, 50, 1)
for example if you use test_features_new to predict on:
- `230668` = number of overlapping sequences
- `50` = time steps per sequence
- `1` = one predicted probability per time step

#### 3. Averaging Across Time Steps
To obtain a single prediction score per sequence, we take the **mean over the 50 time steps**:

this is what this line does:

p_pred = np.mean(probs_pred, axis=1)  # shape: (230668, 1)


#### 4. Flattening for Evaluation + Writing to Dataframes

We then remove the extra singleton dimension to make a 1D array using the folliwing line:

p_pred = np.squeeze(p_pred)  # shape: (230668,)

If we do not use squeeze, the column becomes a column of 1D arrays, not floats which makes plotting etc harder since you have array [0.8] instead of 0.8.

In [44]:
probs_test = model4.predict(test_features_new, batch_size=1024)   # shape: (num_sequences, 50, 1)
avg_scores = np.mean(probs_test, axis=1)                          # shape: (num_sequences, 1)
p_pred = np.squeeze(avg_scores)                                   # shape: (num_sequences,)
y_pred = (p_pred > 0.6).astype(int)

226/226 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step


In [45]:
print(probs_test.shape) #check shape prior to averaging accross time steps

(230668, 50, 1)


In [46]:
print(avg_scores.shape) # check shape after averaging over time steps

(230668, 1)


In [53]:
print(y_pred.shape) #check shape after using squeeze

(230668,)


In [49]:
print(test_labels_new.shape) # checking shaoe of test_labels - this needs to

(230668, 50)


In [50]:
# Get soft labels per sequence (fraction of time steps labeled 1)
y_labels_seq = np.mean(test_labels_new, axis=1)  # shape: (230668,)

# Optionally binarize to match thresholded predictions
y_labels_seq = (y_labels_seq > 0.5).astype(int)   # or keep soft predictions from  model if computing ROC


In [54]:
print(y_labels_seq.shape) # making sure it is the same shape as y_pred

(230668,)


Now we can calculate metrics after checking that all shapes match!

PLOTTING METRICS

ROC CURVE

In [55]:
fpr, tpr, thresholds = roc_curve(y_labels_seq, y_pred)
roc_auc = auc(fpr, tpr)

In [ ]:
plt.figure()
plt.plot(fpr, tpr, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], 'k--', label='No Skill')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for SwitchBack Classification')
plt.legend()
plt.show()

PRC CURVE

In [32]:
precision, recall, thresholds = precision_recall_curve(y_labels_seq, y_pred)
auc_score = auc(recall, precision)

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, label=f'Precision-Recall Curve (AUC = {auc_score:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.show()

additional plots to use if you choose to do so

training history plot

In [34]:
def plot_metrics(history):
  metrics = ['loss', 'precision', 'recall', 'accuracy']
  for n, metric in enumerate(metrics):
    name = metric.replace("_"," ").capitalize()
    plt.subplot(2,2,n+1)
    plt.plot(history.epoch, history.history[metric], label='Train')
    plt.plot(history.epoch, history.history['val_'+metric],
              linestyle="--", label='Val')
    plt.xlabel('Epoch')
    plt.ylabel(name)
    if metric == 'loss':
      plt.ylim([0, plt.ylim()[1]])
    elif metric == 'auc':
      plt.ylim([0.8,1])
    else:
      plt.ylim([0,1])

    plt.legend()

In [ ]:
#Usage
plot_metrics(history_model4)

Confusion Matrix plot

In [35]:

def plot_cm(labels, predictions, threshold=0.6):
  cm = confusion_matrix(labels, predictions > threshold)
  plt.figure(figsize=(5,5))
  sns.heatmap(cm, annot=True, fmt="d")
  plt.title('Confusion matrix @{:.2f}'.format(threshold))
  plt.ylabel('Actual label')
  plt.xlabel('Predicted label')

  print(' (True Negatives): ', cm[0][0])
  print(' (False Positives): ', cm[0][1])
  print('(False Negatives): ', cm[1][0])
  print(' (True Positives): ', cm[1][1])
  print('Total  SB: ', np.sum(cm[1]))

In [ ]:
#Usage if using hard predictions after thresholding
plot_cm(y_labels_seq, y_pred)


In [ ]:
#Usage if using soft predictions (no thresholding)
plot_cm(y_labels_seq, p_pred, threshold = 0.6)

In [ ]:
#EXTRA NOTES
# IF USING NO OVERLAPPING WINDOWS
#probs_test = model4.predict(test_features_new, batch_size=1024)   # shape: (num_sequences, 50, 1)
#p_pred = np.squeeze(probs_test, axis=2)
#p_flat = p_pred.flatten()
#test_label = test_labels_new.flatten()
#Then you can use calculate precision_recall_curve

**Final notes to user: Now that you know how to train ParkerNet, you may retrain ParkerNet with different data. Just make sure the variables, variable order, and time resolution are the same as in this notebook. **